# Entraînement et évaluation des modèles de Deep Learning

## Objectif

Après l'étude des modèles de Machine Learning, cette partie est consacrée à l'utilisation des réseaux de neurones artificiels pour la détection des intrusions dans le jeu de données CICIDS2017.

Les modèles de Deep Learning sont capables d'apprendre automatiquement des représentations complexes des données grâce à plusieurs couches de neurones. Ils sont particulièrement adaptés aux problèmes de classification comportant un grand nombre de variables et de classes.

Dans cette étude, deux architectures sont évaluées :

- **MLP (Multi-Layer Perceptron)** : réseau de neurones entièrement connecté composé d'une couche d'entrée, d'une ou plusieurs couches cachées et d'une couche de sortie.
- **DNN (Deep Neural Network)** : réseau de neurones profond comportant plusieurs couches cachées permettant d'apprendre des relations plus complexes entre les variables.

Les performances seront évaluées à l'aide des mêmes métriques que celles utilisées pour les modèles de Machine Learning afin de garantir une comparaison équitable :

- Accuracy
- Precision
- Recall
- F1-score
- Matrice de confusion
- Rapport de classification

## Importation des bibliothèques

In [19]:
import sys
import numpy as np

print("Version de Python :", sys.version)
print("Version de NumPy :", np.__version__)

!{sys.executable} -m pip show tensorflow

Version de Python : 3.11.2 (v3.11.2:878ead1ac1, Feb  7 2023, 10:02:41) [Clang 13.0.0 (clang-1300.0.29.30)]
Version de NumPy : 2.4.6
Name: tensorflow
Version: 2.16.2
Summary: TensorFlow is an open source machine learning framework for everyone.
Home-page: https://www.tensorflow.org/
Author: Google Inc.
Author-email: packages@tensorflow.org
License: Apache 2.0
Location: /Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages
Requires: absl-py, astunparse, flatbuffers, gast, google-pasta, grpcio, h5py, keras, libclang, ml-dtypes, numpy, opt-einsum, packaging, protobuf, requests, setuptools, six, tensorboard, tensorflow-io-gcs-filesystem, termcolor, typing-extensions, wrapt
Required-by: 


In [20]:
import sys
!{sys.executable} -m pip install "numpy<2.0" --force-reinstall

  Using cached numpy-1.26.4-cp311-cp311-macosx_10_9_x86_64.whl (20.6 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip available: 22.3.1 -> 26.1.2
[notice] To update, run: python3.11 -m pip install --upgrade pip


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

print("TensorFlow version :", tf.__version__)
print("NumPy version :", np.__version__)

2026-07-23 05:09:11.392374: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


TensorFlow version : 2.16.2
NumPy version : 1.26.4


## Chargement des données

Les ensembles d'entraînement et de test préparés lors des étapes précédentes sont chargés afin de construire les modèles de Deep Learning.

In [2]:
X_train = pd.read_csv("./data/processed/X_train_balanced.csv")
X_test = pd.read_csv("./data/processed/X_test_scaled.csv")

y_train = pd.read_csv("./data/processed/y_train_balanced.csv").squeeze()
y_test = pd.read_csv("./data/processed/y_test.csv").squeeze()

## Vérification des données

Avant l'entraînement des modèles, les dimensions des ensembles d'entraînement et de test sont vérifiées.

In [3]:
print("X_train :", X_train.shape)
print("X_test :", X_test.shape)

print("y_train :", y_train.shape)
print("y_test :", y_test.shape)

X_train : (703347, 47)
X_test : (504473, 47)
y_train : (703347,)
y_test : (504473,)


## Encodage des étiquettes

Les réseaux de neurones utilisent une fonction d'activation **Softmax** pour la classification multiclasse.

Par conséquent, les classes doivent être converties en vecteurs binaires à l'aide du **One-Hot Encoding**.

In [5]:
from tensorflow.keras.utils import to_categorical

In [6]:
nb_classes = len(np.unique(y_train))

y_train_cat = to_categorical(
    y_train,
    num_classes=nb_classes
)

y_test_cat = to_categorical(
    y_test,
    num_classes=nb_classes
)

print(y_train_cat.shape)

(703347, 15)


## Vérification de l'encodage

Les nouvelles dimensions des étiquettes sont vérifiées afin de confirmer que chaque observation est désormais représentée sous la forme d'un vecteur binaire.

In [7]:
print(y_train_cat[:5])

[[1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]]


# 1. Multi-Layer Perceptron (MLP)

## Présentation

Le Multi-Layer Perceptron (MLP) est un réseau de neurones artificiel composé d'une couche d'entrée, d'une ou plusieurs couches cachées entièrement connectées et d'une couche de sortie.

Grâce aux fonctions d'activation non linéaires, le MLP est capable d'apprendre des relations complexes entre les variables du jeu de données et constitue l'un des modèles de Deep Learning les plus utilisés pour les problèmes de classification.

## Construction du modèle MLP

Le modèle **Multi-Layer Perceptron (MLP)** est constitué d'une couche d'entrée, de deux couches cachées entièrement connectées et d'une couche de sortie.

Les couches cachées utilisent la fonction d'activation **ReLU (Rectified Linear Unit)** afin d'introduire de la non-linéarité dans le réseau. Des couches de **Dropout** sont également intégrées afin de limiter le phénomène de surapprentissage (*overfitting*) en désactivant aléatoirement une partie des neurones durant l'entraînement.

La couche de sortie utilise la fonction d'activation **Softmax**, adaptée aux problèmes de classification multiclasse. Elle permet de produire une distribution de probabilités sur l'ensemble des classes.

In [8]:
mlp = Sequential([
    
    Dense(
        128,
        activation="relu",
        input_shape=(X_train.shape[1],)
    ),

    Dropout(0.3),

    Dense(
        64,
        activation="relu"
    ),

    Dropout(0.3),

    Dense(
        nb_classes,
        activation="softmax"
    )
])

mlp.summary()

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │         6,144 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 15)             │           975 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 15,375 (60.06 KB)

 Trainable params: 15,375 (60.06 KB)

 Non-trainable params: 0 (0.00 B)

## Compilation du modèle

Avant son entraînement, le modèle est compilé en définissant :

- **Adam** comme algorithme d'optimisation ;
- **Categorical Crossentropy** comme fonction de perte, adaptée aux problèmes de classification multiclasse avec un encodage One-Hot ;
- **Accuracy** comme métrique principale d'évaluation durant l'apprentissage.

In [9]:
mlp.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

## Entraînement du modèle

Le modèle est entraîné sur l'ensemble d'entraînement équilibré.

L'apprentissage est réalisé pendant **20 époques** avec une taille de lot (*batch size*) de **256**.

Une partie des données d'entraînement (**20 %**) est réservée pour la validation afin de suivre les performances du modèle pendant l'entraînement.

In [10]:
history = mlp.fit(
    X_train,
    y_train_cat,
    epochs=20,
    batch_size=256,
    validation_split=0.2,
    verbose=1
)

Epoch 1/20
2198/2198 ━━━━━━━━━━━━━━━━━━━━ 16s 6ms/step - accuracy: 0.9511 - loss: 0.1633 - val_accuracy: 0.1292 - val_loss: 20.3736
Epoch 2/20
2198/2198 ━━━━━━━━━━━━━━━━━━━━ 15s 7ms/step - accuracy: 0.9785 - loss: 0.0606 - val_accuracy: 0.1681 - val_loss: 28.0096
Epoch 3/20
2198/2198 ━━━━━━━━━━━━━━━━━━━━ 15s 7ms/step - accuracy: 0.9826 - loss: 0.0484 - val_accuracy: 0.1703 - val_loss: 37.2472
Epoch 4/20
2198/2198 ━━━━━━━━━━━━━━━━━━━━ 19s 8ms/step - accuracy: 0.9851 - loss: 0.0431 - val_accuracy: 0.1572 - val_loss: 48.5262
Epoch 5/20
2198/2198 ━━━━━━━━━━━━━━━━━━━━ 19s 9ms/step - accuracy: 0.9869 - loss: 0.0379 - val_accuracy: 0.1729 - val_loss: 58.7282
Epoch 6/20
2198/2198 ━━━━━━━━━━━━━━━━━━━━ 21s 9ms/step - accuracy: 0.9878 - loss: 0.0353 - val_accuracy: 0.1728 - val_loss: 68.4812
Epoch 7/20
2198/2198 ━━━━━━━━━━━━━━━━━━━━ 25s 11ms/step - accuracy: 0.9889 - loss: 0.0338 - val_accuracy: 0.1728 - val_loss: 79.5021
Epoch 8/20
2198/2198 ━━━━━━━━━━━━━━━━━━━━ 20s 9ms/step - accuracy: 0.9894 -

## Prédiction

Une fois l'entraînement terminé, le modèle est utilisé pour prédire les classes des observations appartenant à l'ensemble de test.

In [13]:
y_pred_prob = mlp.predict(X_test)

y_pred = np.argmax(y_pred_prob, axis=1)

15765/15765 ━━━━━━━━━━━━━━━━━━━━ 35s 2ms/step


## Évaluation du modèle

Les performances du modèle sont évaluées à l'aide des métriques suivantes :

- Accuracy
- Precision
- Recall
- F1-score

Ces métriques sont calculées selon une moyenne pondérée (*weighted average*) afin de prendre en compte le caractère multiclasse du problème.

In [15]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

In [16]:
accuracy = accuracy_score(y_test, y_pred)

precision = precision_score(
    y_test,
    y_pred,
    average="weighted",
    zero_division=0
)

recall = recall_score(
    y_test,
    y_pred,
    average="weighted",
    zero_division=0
)

f1 = f1_score(
    y_test,
    y_pred,
    average="weighted",
    zero_division=0
)

print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-score  : {f1:.4f}")

Accuracy  : 0.9548
Precision : 0.9185
Recall    : 0.9548
F1-score  : 0.9362
